In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "/Users/andreweross/Documents/GitHub/GDP_inflation_prediction/master_US.csv"

df = pd.read_csv(CSV_PATH)
df["DATE"] = pd.to_datetime(df["DATE"])
df = df.sort_values("DATE").reset_index(drop=True)

# Keep all columns originally
data = df.copy()

print(data.head())

print(data.info())

In [ ]:
# create tariff rate raw number, only quarterly though
data["tariff_rate_raw"] = data["Customs_Duties"] / data["Imports_Goods"]

#filter data just for after 2000
current_data = data.where(data["DATE"] >= "1999-01-01")

#some command so that we dont fill with all nas
current_data = current_data.dropna(subset=["DATE"]).reset_index(drop=True)

# fill in blanks using linear method
current_data["tariff_rate"] = current_data["tariff_rate_raw"].interpolate(method = "linear")
current_data["Customs_Duties"] = current_data["Customs_Duties"].interpolate(method = "linear")
current_data["Imports_Goods"] = current_data["Imports_Goods"].interpolate(method = "linear")
current_data["GDP_nominal"] = current_data["GDP_nominal"].interpolate(method = "linear")
current_data["Real GDP"] = current_data["Real GDP"].interpolate(method = "linear")

print(current_data.head())

In [ ]:
forecast_horizon = 12
# we want to predict the CPI 12 months from now (valuable for businesses)
current_data["cpi_future"] = current_data["CPI"].shift(-forecast_horizon)

#create shifted tariff rates - so in month a, the row has the current tariff rate / 
#tariff rate 3 months before, 6 months before, and 12 months before /
# this enables that we can use old tariff rates to predict current CPIs
current_data["tau_lag_3"] = current_data["tariff_rate"].shift(3)
current_data["tau_lag_6"] = current_data["tariff_rate"].shift(6)
current_data["tau_lag_12"] = current_data["tariff_rate"].shift(12)

# repeat with CPI so we can use old price levels to predict current price levels
current_data["cpi_lag_3"] = current_data["CPI"].shift(1)
current_data["cpi_lag_6"] = current_data["CPI"].shift(3)
current_data["cpi_lag_12"] = current_data["CPI"].shift(12)

#filter data just for after 2020
century_data = current_data.where(current_data["DATE"] >= "2000-01-01")
century_data = century_data.dropna(subset=["DATE"]).reset_index(drop=True)

print(century_data.head())

In [ ]:
target_col = "cpi_future"

exclude_cols = ["DATE", target_col, "tariff_rate_raw"]
feature_cols = [col for col in century_data.columns if col not in exclude_cols]

X = century_data[feature_cols]
y = century_data[target_col]

print("Number of features:", len(feature_cols))
print(feature_cols)

In [ ]:
n = len(century_data)

#create 70 train, 20 validation, 10 test
train_end = int(n * 0.7)
val_end = int(n * 0.9)

#for feature columns
X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y.iloc[:train_end].copy()
y_val = y.iloc[train_end:val_end].copy()
y_test = y.iloc[val_end:].copy()

#for date column
dates_train = century_data["DATE"].iloc[:train_end]
dates_val = century_data["DATE"].iloc[train_end:val_end]
dates_test = century_data["DATE"].iloc[val_end:]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#scale values
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("X Train shape:", X_train_scaled.shape)
print("X Validation shape:", X_val_scaled.shape)
print("X Test shape:", X_test_scaled.shape)
print("y Train shape:", y_train.shape)
print("y Validation shape:", y_val.shape)
print("y Test shape:", y_test.shape)

print("Train range:", dates_train.iloc[0], "to", dates_train.iloc[-1])
print("Validation range:", dates_val.iloc[0], "to", dates_val.iloc[-1])
print("Test range:", dates_test.iloc[0], "to", dates_test.iloc[-1])

In [ ]:
#%pip install xgboost
from xgboost import XGBRegressor

# random parameters - may tweak later
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# predict values for validation set, compute MAE, RMSE, R2
y_val_pred = xgb_model.predict(X_val)

print("Validation MAE:", mean_absolute_error(y_val, y_val_pred))
print("Validation RMSE:", np.sqrt(mean_squared_error(y_val, y_val_pred)))
print("Validation R^2:", r2_score(y_val, y_val_pred))


In [ ]:

val_results = century_data.iloc[train_end:val_end][["DATE"]].copy()
val_results["actual_cpi"] = y_val.values
val_results["predicted_cpi"] = y_val_pred
val_results["error"] = val_results["predicted_cpi"] - val_results["actual_cpi"]
val_results["abs_error"] = val_results["error"].abs()

print(val_results.to_string(index=False))


In [ ]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.to_string(index=False))


In [ ]:
from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

In [ ]:
y_val_pred_lasso = lasso_model.predict(X_val_scaled)

print("Validation MAE:", mean_absolute_error(y_val, y_val_pred_lasso))
print("Validation RMSE:", np.sqrt(mean_squared_error(y_val, y_val_pred_lasso)))
print("Validation R^2:", r2_score(y_val, y_val_pred_lasso))

In [ ]:
val_results_lasso = century_data.iloc[train_end:val_end][["DATE"]].copy()
val_results_lasso["actual_cpi"] = y_val.values
val_results_lasso["predicted_cpi"] = y_val_pred_lasso
val_results_lasso["error"] = val_results_lasso["predicted_cpi"] - val_results_lasso["actual_cpi"]
val_results_lasso["abs_error"] = val_results_lasso["error"].abs()

print(val_results_lasso.to_string(index=False))

In [ ]:
lasso_importance = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "coefficient": lasso_model.coef_,
    "abs_coefficient": np.abs(lasso_model.coef_)
}).sort_values("abs_coefficient", ascending=False)

lasso_importance